## **1) Recherche colonnes avec trop peu d'infos ou pas infos aberrantes** : 

### **1.1) Colonnes avec trop peu d'infos :**

In [1]:
import pandas as pd

df1 = pd.read_csv('/home/michel/Documents/LaPlateforme/Projets/annee2/Data_Wiz_Dash/dazzle_dash/data/life_expectancy_data.csv')
df = df1.columns.str.strip()



## **Mise en graphe avec curseur**

### Code pour graphe avec curseur, gestion automatique des données abberantes et des  valeurs de l'axe Y :

In [3]:
import plotly.graph_objects as go


def calculer_traces(d, colonne, seuil):
    """Calcule les 4 traces du graphique pour un seuil donné.
    d : lignes d'UN seul pays, triées par année."""
    # Une valeur est suspecte si elle est < seuil x médiane du pays
    flag = d[colonne] < seuil * d[colonne].median()

    # Valeurs d'origine indexées par année
    origine = d.set_index("Year")[colonne]

    # Valeurs suspectes effacées (NaN), année en index pour l'interpolation
    valeurs = d[colonne].where(~flag)
    valeurs.index = d["Year"].values

    # Interpolation linéaire uniquement ENTRE deux valeurs retenues
    interpolees = valeurs.interpolate(method="index", limit_area="inside")

    # Années flaggées : interpolables (rouge) ou non (orange, bord de série)
    annees_suspectes = d.loc[flag, "Year"]
    points_rouges = interpolees.reindex(annees_suspectes).dropna()
    non_interpolables = annees_suspectes[~annees_suspectes.isin(points_rouges.index)]
    retenues = valeurs.dropna()

    # Années où la valeur d'origine est ABSENTE (NaN) mais remplie par l'interpolation
    annees_nan = d.loc[d[colonne].isna(), "Year"]
    points_nan = interpolees.reindex(annees_nan).dropna()  # dropna : on retire les NaN non interpolables

    bleu = "#636EFA"
    return [
        # Trace 1 : la ligne seule, sans marqueurs ni infobulle
        go.Scatter(x=interpolees.index, y=interpolees.values, mode="lines",
                   line=dict(color=bleu), hoverinfo="skip", showlegend=False),
        # Trace 2 : marqueurs bleus uniquement sur les valeurs retenues
        go.Scatter(x=retenues.index, y=retenues.values, mode="markers",
                   name="Valeurs retenues", marker=dict(color=bleu)),
        # Trace 3 : points rouges, avec la valeur d'origine dans l'infobulle
        go.Scatter(x=points_rouges.index, y=points_rouges.values, mode="markers",
                   name="Valeurs interpolées", marker=dict(color="red", size=10),
                   customdata=origine.loc[points_rouges.index].values,
                   hovertemplate=("Année : %{x}<br>"
                                  "Valeur interpolée : %{y:.1f}<br>"
                                  "Valeur d'origine : %{customdata}<extra></extra>")),
        # Trace 4 : suspectes non remplaçables, laissées à leur valeur d'origine
        go.Scatter(x=non_interpolables.values, y=origine.loc[non_interpolables].values,
                   mode="markers", name="Suspectes non remplacées (bord de série)",
                   marker=dict(color="orange", size=10, symbol="x"),
                   hovertemplate="Année : %{x}<br>Valeur d'origine : %{y}<extra></extra>"),
    
        # Trace 5 : valeurs absentes (NaN) comblées par interpolation (losanges violets)
        go.Scatter(x=points_nan.index, y=points_nan.values, mode="markers",
                   name="Valeurs absentes interpolées",
                   marker=dict(color="purple", size=10, symbol="diamond"),
                   hovertemplate=("Année : %{x}<br>"
                                  "Valeur interpolée : %{y:.1f}<br>"
                                  "Valeur d'origine : absente<extra></extra>")),     
        ] 

def courbe_avec_outliers(df, pays, colonne, titre, seuil_initial=0.5, pas=0.05):
    """Courbe d'un pays avec curseur de seuil placé sous la légende."""
    d = df[df["Country"] == pays].sort_values("Year").copy()

    # Liste des seuils proposés par le curseur : 0, pas, 2*pas, ..., 1
    seuils = [round(i * pas, 2) for i in range(int(round(1 / pas)) + 1)]
    indice_initial = seuils.index(seuil_initial)

    # On calcule les traces de TOUS les seuils à l'avance
    # (le curseur ne fait ensuite qu'afficher/masquer des traces)
    traces_par_seuil = [calculer_traces(d, colonne, s) for s in seuils]
    nb = len(traces_par_seuil[0])  # 4 traces par seuil
    
    # Axe des ordonnées : de 0 à 10 % au-dessus du maximum de la colonne pour ce pays
    maximum = d[colonne].max()  # NaN si le pays n'a aucune valeur
    plage_y = [0, maximum * 1.1] if pd.notna(maximum) and maximum > 0 else None  # None = automatique
    
    fig = go.Figure()
    for i, traces in enumerate(traces_par_seuil):
        for t in traces:
            t.visible = (i == indice_initial)  # seul le seuil initial est visible
            fig.add_trace(t)

    # Une étape du curseur = rendre visibles les 4 traces de ce seuil, masquer les autres
    etapes = []
    for i, s in enumerate(seuils):
        visibilite = [False] * (nb * len(seuils))
        for k in range(nb):
            visibilite[i * nb + k] = True
        etapes.append(dict(method="restyle", args=[{"visible": visibilite}], label=f"{s:.2f}"))

    fig.update_layout(
        title=titre, xaxis_title="Année",
        yaxis=dict(title=colonne, range=plage_y),
        width=1100, height=500, margin=dict(r=380),   # marge droite pour légende + curseur
        legend=dict(x=1.02, y=1, xanchor="left", yanchor="top"),
        sliders=[dict(
            active=indice_initial, steps=etapes,
            x=1.02, xanchor="left", y=0.72, yanchor="top", len=0.3,  # sous la légende
            currentvalue=dict(prefix="Seuil de multiplication de la médiane : ", xanchor="left",
                              font=dict(color="#2a3f5f", size=14)),
            font=dict(color="rgba(0,0,0,0)"),  # masque les 21 étiquettes sous le curseur
            ticklen=0,
        )],
    )
    return fig


courbe_avec_outliers(df, "Chad", "Adult Mortality", "Mortalité adulte").show()
courbe_avec_outliers(df, "France", "Alcohol", "Consommation d'alcool  par habitants, en litres").show()


IndexError: only integers, slices (`:`), ellipsis (`...`), numpy.newaxis (`None`) and integer or boolean arrays are valid indices